# Create MCP Agent using OpenVINO and Qwen-Agent

MCP is an open protocol that standardizes how applications provide context to LLMs. Think of MCP like a USB-C port for AI applications. Just as USB-C provides a standardized way to connect your devices to various peripherals and accessories, MCP provides a standardized way to connect AI models to different data sources and tools.

MCP helps you build agents and complex workflows on top of LLMs. LLMs frequently need to integrate with data and tools, and MCP provides:

- A growing list of pre-built integrations that your LLM can directly plug into
- The flexibility to switch between LLM providers and vendors
- Best practices for securing your data within your infrastructure

![Image](https://github.com/user-attachments/assets/bdb1790a-b464-457b-ae57-a88b9f9bae43)

[Qwen-Agent](https://github.com/QwenLM/Qwen-Agent) is a framework for developing LLM applications based on the instruction following, tool usage, planning, and memory capabilities of Qwen. It also comes with example applications such as Browser Assistant, Code Interpreter, and Custom Assistant.

This notebook explores how to create a MCP Agent step by step using OpenVINO and Qwen-Agent.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Download model](#Download-model)
- [Create an Agent](#Create-An-Agent)
- [Interactive Demo](#Interactive-Demo)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).


<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/llm-agent-mcp/llm-agent-mcp.ipynb" />


## Prerequisites

[back to top ⬆️](#Table-of-contents:)


In [1]:
import os
from pathlib import Path
import requests

os.environ["GIT_CLONE_PROTECTION_ACTIVE"] = "false"

%pip install -Uq pip
%pip uninstall -q -y optimum optimum-intel
%pip install --pre -Uq "openvino>=2024.2.0" openvino-tokenizers[transformers] --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly
%pip install -q --extra-index-url https://download.pytorch.org/whl/cpu \
"torch>=2.1" "datasets" "accelerate" "transformers>=4.51.0"
"pydantic==2.9.2" "pydantic-core==2.23.4" "gradio>=5.0.0" "gradio-client==1.4.0" "modelscope_studio==1.0.0-beta.8"
%pip install -q --extra-index-url https://download.pytorch.org/whl/cpu \
"git+https://github.com/huggingface/optimum-intel.git"
%pip install -q "git+https://github.com/openvinotoolkit/nncf.git"
    
utility_files = ["notebook_utils.py", "cmd_helper.py"]

for utility in utility_files:
    local_path = Path(utility)
    if not local_path.exists():
        r = requests.get(
            url=f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{local_path.name}",
        )
        with local_path.open("w") as f:
            f.write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("llm-agent-mcp.ipynb")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from cmd_helper import clone_repo

clone_repo("https://github.com/QwenLM/Qwen-Agent.git")

%pip install -q -e ./Qwen-Agent/"[gui,code_interpreter,mcp]"

## Download model

[back to top ⬆️](#Table-of-contents:)

Large Language Models (LLMs) are a core component of Agent. In this example, we will demonstrate how to create a OpenVINO LLM model in Qwen-Agent framework. Since Qwen3 can support function calling during text generation, we select `Qwen/Qwen3-8B` as LLM in agent pipeline.

* **Qwen/Qwen3-8B** - Qwen3 is the latest generation of large language models in Qwen series, offering a comprehensive suite of dense and mixture-of-experts (MoE) models. Built upon extensive training, Qwen3 delivers groundbreaking advancements in reasoning, instruction-following, agent capabilities, and multilingual suppor. [Model Card](https://huggingface.co/Qwen/Qwen3-8B)

To run LLM locally, we have to download the model in the first step. It is possible to [export your model](https://github.com/huggingface/optimum-intel?tab=readme-ov-file#export) to the OpenVINO IR format with the CLI, and load the model from local folder.


In [ ]:
from pathlib import Path
from cmd_helper import optimum_cli

model_id = "Qwen/Qwen3-8B"
model_path = "Qwen/Qwen3-8B-ov"

if not Path(model_path).exists():
    optimum_cli(model_id, model_path, additional_args={"task": "text-generation-with-past", "trust-remote-code": "", "weight-format": "int4", "ratio": "0.8"})

## Configure MCP servers

[back to top ⬆️](#Table-of-contents:)

MCP server can be configured into an [MCP client](https://github.com/modelcontextprotocol/servers?tab=readme-ov-file#using-an-mcp-client). The configuration of MCP server be selected from [public MCP servers list](https://github.com/punkpeye/awesome-mcp-servers), or from your customized MCP server.

## Create An agent

[back to top ⬆️](#Table-of-contents:)

Function calling allows a model to detect when one or more tools should be called and respond with the inputs that should be passed to those tools. In an API call, you can describe tools and have the model intelligently choose to output a structured object like JSON containing arguments to call these tools. The goal of tools APIs is to more reliably return valid and useful tool calls than what can be done using a generic text completion or chat API.

We can take advantage of this structured output, combined with the fact that you can bind multiple tools to a tool calling chat model and allow the model to choose which one to call, to create an agent that repeatedly calls tools and receives results until a query is resolved.

OpenVINO has been integrated into the `Qwen-Agent` framework. You can use following method to create a OpenVINO based LLM for a `Qwen-Agent` pipeline.
Qwen-Agent offers a generic Agent class: the Assistant class, which, when directly instantiated, can handle the majority of Single-Agent tasks. Features:

- It supports role-playing.
- It provides automatic planning and tool calls abilities.
- RAG (Retrieval-Augmented Generation): It accepts documents input, and can use an integrated RAG strategy to parse the documents.

MCP server can be configured into an [MCP client](https://github.com/modelcontextprotocol/servers?tab=readme-ov-file#using-an-mcp-client). The configuration of MCP server be selected from [public MCP servers list](https://github.com/punkpeye/awesome-mcp-servers), or from your customized MCP server. Since the examples of the MCP server in this notebook are in remote, please make sure your system is connected with internet.

In [10]:
%%writefile mcp_test.py

import openvino.properties as props
import openvino.properties.hint as hints
import openvino.properties.streams as streams
from qwen_agent.agents import Assistant

tools = [
    {
        'mcpServers': {  # You can specify the MCP configuration file
            'time': {
                'command': 'uvx',
                'args': ['mcp-server-time', '--local-timezone=Asia/Shanghai']
            },
            'fetch': {
                'command': 'uvx',
                'args': ['mcp-server-fetch']
            }
        }
    },
    'code_interpreter',  # Built-in tools
]


llm_cfg = {
    "ov_model_dir": "Qwen/Qwen3-8B-ov",
    "model_type": "openvino",
    "device": "GPU",
    # (Optional) LLM hyperparameters for generation:
    "generate_cfg": {"top_k": 1, "fncall_prompt_type": "qwen"},
}

bot = Assistant(llm=llm_cfg,
                system_message="/no_think ",
                function_list=tools,
                name='Qwen3 Tool-calling Demo',
                description="I'm a demo using the Qwen3 tool calling. Welcome to add and play with your own tools!")

messages = [{'role': 'user', 'content': 'What time is it?'}]
response_plain_text = ''
for response in bot.run(messages=messages):
    pass
print(response)

Overwriting mcp_test.py


In [9]:
!python mcp_test.py

2025-05-28 17:01:10,144 - mcp_manager.py - 122 - INFO - Initializing MCP tools from mcp servers: ['time', 'fetch']
2025-05-28 17:01:10,159 - mcp_manager.py - 340 - INFO - Initializing a MCP stdio_client, if this takes forever, please check the config of this mcp server: time
2025-05-28 17:01:27,661 - mcp_manager.py - 350 - INFO - No list resources: Method not found
2025-05-28 17:01:27,670 - mcp_manager.py - 340 - INFO - Initializing a MCP stdio_client, if this takes forever, please check the config of this mcp server: fetch
Installed 36 packages in 181ms
2025-05-28 17:01:32,959 - mcp_manager.py - 350 - INFO - No list resources: Method not found
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
2025-05-28 17:01:44,609 - mcp_manager.py - 277 - INFO - There are still tasks in `MCPManager().loop`, force terminatin

<think>

</think>
[TOOL_CALL] 
ime-get_current_time
{"timezone": "Asia/Shanghai"}
[TOOL_RESPONSE] time-get_current_time
{
  "timezone": "Asia/Shanghai",
  "datetime": "2025-05-28T17:01:38+08:00",
  "is_dst": false
}
<think>

</think>

The current time in Asia/Shanghai is 2025-05-28T17:01:38+08:00.


## Interactive Demo

[back to top ⬆️](#Table-of-contents:)

Let's create a interactive agent using [Gradio](https://www.gradio.app/).

In [7]:
from pathlib import Path
from PIL import Image
import requests

openvino_logo = "openvino_logo.png"
openvino_logo_url = "https://cdn-avatars.huggingface.co/v1/production/uploads/1671615670447-6346651be2dcb5422bcd13dd.png"

if not Path(openvino_logo).exists():
    image = Image.open(requests.get(openvino_logo_url, stream=True).raw)
    image.save(openvino_logo)

In [11]:
%%writefile mcp_demo.py

import openvino.properties as props
import openvino.properties.hint as hints
import openvino.properties.streams as streams
from qwen_agent.agents import Assistant
from gradio_helper import OpenVINOUI

tools = [
    {
        'mcpServers': {  # You can specify the MCP configuration file
            'time': {
                'command': 'uvx',
                'args': ['mcp-server-time', '--local-timezone=Asia/Shanghai']
            },
            'fetch': {
                'command': 'uvx',
                'args': ['mcp-server-fetch']
            },
        }
    },
    'code_interpreter',  # Built-in tools
]


llm_cfg = {
    "ov_model_dir": "Qwen/Qwen3-8B-ov",
    "model_type": "openvino",
    "device": "GPU",
    # (Optional) LLM hyperparameters for generation:
    "generate_cfg": {"top_k": 1, "max_new_tokens":4096, "fncall_prompt_type": "qwen"},
}

bot = Assistant(llm=llm_cfg,
                system_message="/no_think ",
                function_list=tools,
                name='OpenVINO MCP Demo',
                description="I'm a demo using the Qwen3 tool calling. Welcome to add and play with your own tools!")

chatbot_config = {
    'prompt.suggestions': [
        'What time is it?',
        "Covert time of Shanghai to New York"
    ],
    'agent.avatar': "openvino_logo.png",
}

demo = OpenVINOUI(
    bot,
    chatbot_config=chatbot_config,
)
demo.run(server_port=8900)

Overwriting mcp_demo.py


In [ ]:
!python mcp_demo.py

Now you can visit [http://127.0.0.1:8900](http://127.0.0.1:8900) to try this demo. 
If you are launching remotely, specify server_name and server_port. EXAMPLE: 

`demo.run(server_name='your server name', server_port='server port in int')`

To kill the process of demo, you can run following command:

on *Windows*

In [19]:
!for /f "tokens=5" %a in ('netstat -aon ^| findstr ":8900 "') do taskkill /f /pid %a

on *Linux*

In [ ]:
!kill -9 $(lsof -t -i :8080)